In [23]:
import pandas as pd
from pathlib import Path
import re

ROOT = Path("src/peacsim/output/optimization")

all_frames = []

for csv_file in ROOT.rglob("grid_search_results.csv"):

    try:
        df = pd.read_csv(csv_file)

        folder = csv_file.parent.name

        df["source_folder"] = folder

        workload_match = re.search(r"_W(\d+)_", folder)
        if workload_match:
            df["workload_name"] = f"W{workload_match.group(1)}"
        else:
            df["workload_name"] = "unknown"

        server_match = re.search(r"exp_servers_(\d+)", folder)
        if server_match:
            df["experiment_name"] = f"exp_servers_{server_match.group(1)}"
        else:
            df["experiment_name"] = "unknown"

        all_frames.append(df)

        print(
            f"Loaded {csv_file} "
            f"({len(df)} rows)"
        )

    except Exception as e:
        print(f"FAILED: {csv_file}")
        print(e)

combined = pd.concat(
    all_frames,
    ignore_index=True
)

combined.to_csv(
    "flexdc_all_data.csv",
    index=False
)

print()
print("====================================")
print("Rows:", len(combined))
print("Columns:", len(combined.columns))
print("Saved: flexdc_all_data.csv")
print("====================================")



Loaded src\peacsim\output\optimization\dataset_AQA_W10_exp_servers_1000_20260604005843\grid_search_results.csv (3384 rows)
Loaded src\peacsim\output\optimization\dataset_AQA_W10_exp_servers_100_20260604005839\grid_search_results.csv (3384 rows)
Loaded src\peacsim\output\optimization\dataset_AQA_W10_exp_servers_500_20260604005841\grid_search_results.csv (3384 rows)
Loaded src\peacsim\output\optimization\dataset_AQA_W10_exp_servers_64_20260604005836\grid_search_results.csv (3384 rows)
Loaded src\peacsim\output\optimization\dataset_AQA_W11_exp_servers_1000_20260604005842\grid_search_results.csv (3384 rows)
Loaded src\peacsim\output\optimization\dataset_AQA_W11_exp_servers_100_20260604005839\grid_search_results.csv (3384 rows)
Loaded src\peacsim\output\optimization\dataset_AQA_W11_exp_servers_500_20260604005841\grid_search_results.csv (3384 rows)
Loaded src\peacsim\output\optimization\dataset_AQA_W11_exp_servers_64_20260604005836\grid_search_results.csv (3384 rows)
Loaded src\peacsim\outpu

In [24]:
df = pd.read_csv("flexdc_all_data.csv")
print(df.head())
print(df.info())

print(df[df["Weight_8"].notna()][
    ["workload_name","experiment_name"]
].drop_duplicates())

   Iteration  Weight_Sample_ID Policy_Name  Node_Count_Control  \
0          1                 0         AQA                True   
1          2                 1         AQA                True   
2          3                 2         AQA                True   
3          4                 3         AQA                True   
4          5                 4         AQA                True   

   Power_Cap_Percentage Flex_Resource_Probabilities  Pbar_ratio  R_ratio  \
0                   NaN                          ""         0.7     0.05   
1                   NaN                          ""         0.7     0.05   
2                   NaN                          ""         0.7     0.05   
3                   NaN                          ""         0.7     0.05   
4                   NaN                          ""         0.7     0.05   

   P_actual_watts  R_actual_watts  ...  SoftPlus_Tracking_Cost  \
0          166818            5915  ...                0.318124   
1          166

In [25]:
import pandas as pd
import json
import re

df = pd.read_csv("flexdc_all_data.csv")

weight_cols = sorted(
    [col for col in df.columns if re.fullmatch(r"Weight_\d+", col)],
    key=lambda x: int(x.split("_")[1])
)

def make_weights_array(row):
    weights = []

    for col in weight_cols:
        value = row[col]

        if pd.isna(value):
            continue

        weights.append(float(value))

    return json.dumps(weights)

df["weights"] = df.apply(make_weights_array, axis=1)

df = df.drop(columns=weight_cols)

df.to_csv("flexdc_all_data_with_weights_array.csv", index=False)

print()
print("====================================")
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Saved: flexdc_all_data_with_weights_array.csv")
print("====================================")

print(df.head())
print(df.info())

print()
print("Weight-vector lengths by workload:")
df["weight_count"] = df["weights"].apply(lambda x: len(json.loads(x)))

print(
    df.groupby("workload_name")["weight_count"]
      .unique()
      .sort_index()
)

print()
print("Example weights:")
print(
    df[["workload_name", "experiment_name", "weights"]]
      .drop_duplicates()
      .head()
)


Rows: 81216
Columns: 30
Saved: flexdc_all_data_with_weights_array.csv
   Iteration  Weight_Sample_ID Policy_Name  Node_Count_Control  \
0          1                 0         AQA                True   
1          2                 1         AQA                True   
2          3                 2         AQA                True   
3          4                 3         AQA                True   
4          5                 4         AQA                True   

   Power_Cap_Percentage Flex_Resource_Probabilities  Pbar_ratio  R_ratio  \
0                   NaN                          ""         0.7     0.05   
1                   NaN                          ""         0.7     0.05   
2                   NaN                          ""         0.7     0.05   
3                   NaN                          ""         0.7     0.05   
4                   NaN                          ""         0.7     0.05   

   P_actual_watts  R_actual_watts  ...  SoftPlus_Tracking_Cost  \
0        

In [26]:
df = pd.read_csv("flexdc_all_data_with_weights_array.csv")
print(df.head())
print(df.info())

   Iteration  Weight_Sample_ID Policy_Name  Node_Count_Control  \
0          1                 0         AQA                True   
1          2                 1         AQA                True   
2          3                 2         AQA                True   
3          4                 3         AQA                True   
4          5                 4         AQA                True   

   Power_Cap_Percentage Flex_Resource_Probabilities  Pbar_ratio  R_ratio  \
0                   NaN                          ""         0.7     0.05   
1                   NaN                          ""         0.7     0.05   
2                   NaN                          ""         0.7     0.05   
3                   NaN                          ""         0.7     0.05   
4                   NaN                          ""         0.7     0.05   

   P_actual_watts  R_actual_watts  ...  SoftPlus_Tracking_Cost  \
0          166818            5915  ...                0.318124   
1          166

In [27]:
import numpy as np
import pandas as pd

# ============================================================
# Fix tracking columns in the existing dataframe
# ============================================================

TRACKING_ERROR_THRESHOLD = 0.3

# ------------------------------------------------------------
# 1. Handle old vs already-fixed tracking-cost column name
# ------------------------------------------------------------

if "Simulator_Tracking_Error" in df.columns and "Simulator_Tracking_Cost" in df.columns:
    raise ValueError(
        "Both Simulator_Tracking_Error and Simulator_Tracking_Cost exist. "
        "Please inspect manually before continuing."
    )

if "Simulator_Tracking_Error" in df.columns:
    df = df.rename(columns={
        "Simulator_Tracking_Error": "Simulator_Tracking_Cost"
    })

# ------------------------------------------------------------
# 2. Validate required columns
# ------------------------------------------------------------

required_cols = [
    "Power_Cost",
    "Raw_QoS_Cost",
    "Raw_Total_Cost",
    "Raw_Tracking_Cost",
    "Simulator_Tracking_Cost",
    "Tracking_Error_90th",
]

missing = [col for col in required_cols if col not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# ------------------------------------------------------------
# 3. Fix Raw_Tracking_Cost
# ------------------------------------------------------------
# Correct paper-style raw tracking term:
# Raw_Tracking_Cost = Tracking_Error_90th - 0.3

df["Raw_Tracking_Cost"] = (
    df["Tracking_Error_90th"] - TRACKING_ERROR_THRESHOLD
)

# ------------------------------------------------------------
# 4. Fix Raw_Total_Cost
# ------------------------------------------------------------

df["Raw_Total_Cost"] = (
    df["Power_Cost"]
    + df["Raw_Tracking_Cost"]
    + df["Raw_QoS_Cost"]
)

# ------------------------------------------------------------
# 5. Drop old SoftPlus columns from this cleaned dataset
# ------------------------------------------------------------
# These were computed from the old incorrect tracking term in the
# already-generated dataset, so remove them from the cleaned training CSV.

softplus_cols = [
    "SoftPlus_Total_Cost",
    "SoftPlus_Tracking_Cost",
    "SoftPlus_QoS_Cost",
]

existing_softplus_cols = [col for col in softplus_cols if col in df.columns]

if existing_softplus_cols:
    df = df.drop(columns=existing_softplus_cols)
    print("Dropped old SoftPlus columns:", existing_softplus_cols)
else:
    print("No SoftPlus columns found to drop.")

# ------------------------------------------------------------
# 6. Sanity checks
# ------------------------------------------------------------

reconstructed_total = (
    df["Power_Cost"]
    + df["Raw_Tracking_Cost"]
    + df["Raw_QoS_Cost"]
)

max_diff = np.abs(df["Raw_Total_Cost"] - reconstructed_total).max()

print("Max difference after Raw_Total_Cost fix:", max_diff)

if max_diff > 1e-9:
    raise ValueError("Raw_Total_Cost reconstruction check failed.")

# Check raw tracking cost formula
tracking_max_diff = np.abs(
    df["Raw_Tracking_Cost"]
    - (df["Tracking_Error_90th"] - TRACKING_ERROR_THRESHOLD)
).max()

print("Max difference after Raw_Tracking_Cost fix:", tracking_max_diff)

if tracking_max_diff > 1e-9:
    raise ValueError("Raw_Tracking_Cost reconstruction check failed.")

print()
print("Final fixed tracking/cost columns:")
print([
    col for col in [
        "Power_Cost",
        "Simulator_Tracking_Cost",
        "Tracking_Error_90th",
        "Raw_Tracking_Cost",
        "Raw_QoS_Cost",
        "Raw_Total_Cost",
    ]
    if col in df.columns
])

print()
print(df[[
    "Power_Cost",
    "Simulator_Tracking_Cost",
    "Tracking_Error_90th",
    "Raw_Tracking_Cost",
    "Raw_QoS_Cost",
    "Raw_Total_Cost",
]].head())

# ------------------------------------------------------------
# 7. Save fixed CSV
# ------------------------------------------------------------

output_path = "flexdc_all_data_fixed.csv"

df.to_csv(output_path, index=False)

print()
print("====================================")
print("Fixed CSV saved:", output_path)
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("====================================")

Dropped old SoftPlus columns: ['SoftPlus_Total_Cost', 'SoftPlus_Tracking_Cost', 'SoftPlus_QoS_Cost']
Max difference after Raw_Total_Cost fix: 0.0
Max difference after Raw_Tracking_Cost fix: 0.0

Final fixed tracking/cost columns:
['Power_Cost', 'Simulator_Tracking_Cost', 'Tracking_Error_90th', 'Raw_Tracking_Cost', 'Raw_QoS_Cost', 'Raw_Total_Cost']

   Power_Cost  Simulator_Tracking_Cost  Tracking_Error_90th  \
0     16.0903                 0.025405                0.109   
1     16.0903                 0.020425                0.090   
2     16.0903                 0.029142                0.128   
3     16.0903                 0.048747                0.223   
4     16.0903                 0.023196                0.098   

   Raw_Tracking_Cost  Raw_QoS_Cost  Raw_Total_Cost  
0             -0.191      5.387142       21.286442  
1             -0.210      5.352576       21.232876  
2             -0.172      5.277809       21.196109  
3             -0.077      4.505819       20.519119  
4    

In [ ]:
df = pd.read_csv("flexdc_all_data_fixed.csv")
print(df.head())